In [3]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Load dataset
df = pd.read_csv("uber.csv")

print(df.head())
print("Original Dataset Shape:", df.shape)

# Data preprocessing
df = df.dropna()

# Remove invalid and out-of-range values
df = df[
    (df["fare_amount"] > 0) &
    (df["fare_amount"] < 100) &
    (df["passenger_count"].between(1, 6)) &
    (df["pickup_latitude"].between(-90, 90)) &
    (df["dropoff_latitude"].between(-90, 90)) &
    (df["pickup_longitude"].between(-180, 180)) &
    (df["dropoff_longitude"].between(-180, 180))
].copy()

# Convert pickup_datetime to datetime
df["pickup_datetime"] = pd.to_datetime(
    df["pickup_datetime"], errors="coerce", utc=True
)

df = df.dropna(subset=["pickup_datetime"])

# Extract date and time features
df["pickup_hour"] = df["pickup_datetime"].dt.hour
df["pickup_day"] = df["pickup_datetime"].dt.day
df["pickup_month"] = df["pickup_datetime"].dt.month
df["pickup_dayofweek"] = df["pickup_datetime"].dt.dayofweek

# Drop unnecessary columns
df = df.drop(columns=["key", "pickup_datetime"], errors="ignore")

print("\nDataset after preprocessing:")
print(df.shape)
print(df.head())

# Detect outliers using the IQR method
Q1 = df["fare_amount"].quantile(0.25)
Q3 = df["fare_amount"].quantile(0.75)

IQR = Q3 - Q1
lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

outliers = df[
    (df["fare_amount"] < lower_limit) |
    (df["fare_amount"] > upper_limit)
]

print("\nNumber of outliers:", len(outliers))
print("Lower limit:", lower_limit)
print("Upper limit:", upper_limit)

# Visualize fare outliers
plt.figure(figsize=(8, 5))
plt.boxplot(df["fare_amount"])
plt.title("Uber Fare Outliers")
plt.ylabel("fare_amount")
plt.grid(True)
plt.show()

# Separate features and target
X = df.drop("fare_amount", axis=1)
y = df["fare_amount"]

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Standardize features
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Define regression models
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Lasso Regression": Lasso(alpha=0.1, max_iter=10000)
}

# Train models and evaluate performance
results = []

for name, model in models.items():
    model.fit(X_train_scaled, y_train)

    predictions = model.predict(X_test_scaled)

    r2 = r2_score(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    mae = mean_absolute_error(y_test, predictions)

    results.append({
        "Model": name,
        "R2 Score": r2,
        "RMSE": rmse,
        "MAE": mae
    })

# Display comparison of models
results_df = pd.DataFrame(results)

print("\nModel Evaluation Results:")
print(results_df.round(4).to_string(index=False))

# Predict fare for a sample test record
sample = X_test_scaled[0].reshape(1, -1)

print("\nActual Fare:", round(y_test.iloc[0], 2))

for name, model in models.items():
    prediction = model.predict(sample)[0]
    print(name, "Predicted Fare:", round(prediction, 2))


FileNotFoundError: [Errno 2] No such file or directory: 'uber.csv'